# A Shallow Shelf Approximation stabilization scheme

- This code simulates the development of an ice stream using the Shallow Shelf Approximation (SSA) with spatially varying friction.
- We introduce the Thickness Stabilization Scheme (TSS), which allows greater stability and larger stable time steps.
- The core aspects of the TSS-stabilized model code for grounded ice are introduced here.
- A full model description is to be found in *A numerical stabilization method for the shallow shelf approximation* by Tilda Westling Dolling, Clara Henry, and Josefin Ahlkrona [in review].

## The Shallow Shelf Approximation (SSA)
### The SSA strong form
The Shallow Shelf Approximation reads
$$
\begin{aligned}
    &4 \frac{\partial}{\partial x}\biggl(\bar\eta \frac{\partial u_x}{\partial x} \biggl) + 2\frac{\partial}{\partial x}\biggl( \bar\eta \frac{\partial u_y}{\partial y}  \biggl)
    +\frac{\partial}{\partial y}\biggl(\bar\eta \biggl(\frac{\partial u_x}{\partial y} 
    + \frac{\partial u_y}{\partial x} \biggl)\biggl) 
    =\rho_i g H\frac{\partial z_s}{\partial x}  \quad \text{in} \, \Omega(x,y), \\
    &4 \frac{\partial}{\partial y}\biggl(\bar\eta \frac{\partial u_y}{\partial y} \biggl) + 2\frac{\partial}{\partial y}\biggl( \bar\eta \frac{\partial u_x}{\partial x}  \biggl)
    + \frac{\partial}{\partial x}\biggl(\bar\eta \biggl(\frac{\partial u_x}{\partial y} 
    + \frac{\partial u_y}{\partial x} \biggl)\biggl) 
    = \rho_i g H\frac{\partial z_s}{\partial y}\quad \text{in} \, \Omega(x,y),
\end{aligned}\tag{1}
$$
where $\mathbf{u} = (u_x, u_y)$ is the horizontal velocity vector, $H$ is the ice thickness, $z_s$ is the upper ice surface elevation, $\rho_i$ is the ice density, $\rho_o$ is the ocean density, and $g$ is the gravitational constant. 

The vertically-averaged viscosity is
$$
\bar\eta = \frac{1}{2} A^{-1/n} H \left ( \frac{1}{4} \left ( \frac{\partial u_x}{\partial y} + \frac{\partial u_y}{\partial x} \right )^2 + \frac{\partial u_x}{\partial x} \frac{\partial u_y}{\partial y} + \left ( \frac{\partial u_x}{\partial x} \right )^2 + \left ( \frac{\partial u_y}{\partial y} \right )^2  + \dot{\varepsilon}_0^2 \right )^{\frac{1 - n}{2n}},\tag{2}
$$
where $A$ is the ice fluidity, $\dot{\boldsymbol{\varepsilon}}$ is the strain rate tensor, $\|\cdot|^2_F$ is the Frobenius norm and $\dot{\varepsilon}_0^2$ is a regularization term.
### The SSA weak form
For simplicity, we define the tensor
$$
\mathbf{T} = \begin{pmatrix} \bar\eta  \big( 4 \frac{\partial u_x}{\partial x} + 2 \frac{\partial u_y}{\partial y} \big) & \bar\eta  \big( \frac{\partial u_x}{\partial y} + \frac{\partial u_y}{\partial x} \big) \\ \bar\eta  \big( \frac{\partial u_x}{\partial y} + \frac{\partial u_y}{\partial x} \big) & \bar\eta  \big( 2 \frac{\partial u_x}{\partial x} + 4 \frac{\partial u_y}{\partial y} \big) \end{pmatrix},\tag{3}
$$
so that the SSA momentum equations can be rewritten as
$$
    \nabla \cdot \mathbf{T} - \beta \mathbf{u} = \rho_i g H \nabla z_s,\tag{4}
$$
where $\beta$ is a basal friction coefficient.

In weak form, the SSA momentum equations read
$$
\int_{\Omega} (\nabla \cdot \mathbf{T} ) \cdot \mathbf{v} \,d\Omega - \int_{\Omega} \beta^2 \mathbf{u} \cdot \mathbf{v} \,d\Omega = \int_{\Omega} \rho_i g H \nabla z_s \cdot \mathbf{v} \,d\Omega,\tag{5}
$$
where $\mathbf{v} \in \boldsymbol{\chi}$ is a test function in an appropriate Sobolov space.

Substituting in $z_s = z_b + H$, using the relationship $H \nabla H = 1/2 \nabla H^2$ and integrating by parts results in
$$
\int _{\Omega} \mathbf{T} : \nabla \mathbf{v}\,d\Omega - \int_{\partial \Omega} \mathbf{T} \mathbf{n} \cdot \mathbf{v} \,d\Gamma + \int_{\Omega} \beta \mathbf{u} \cdot \mathbf{v} \,d\Omega \\
    = - \int_{\Omega}  \rho_i g H \nabla z_b \cdot \mathbf{v} \,d\Omega + \int_{\Omega} \frac{1}{2} \rho_i g H^2 \nabla \cdot \mathbf{v} \,d\Omega - \int_{\partial \Omega} \frac{1}{2} \rho_i g H^2 \mathbf{n} \cdot \mathbf{v} \,d\Omega.\tag{6}
$$

By using Dirichlet boundary conditions on three side of the square domain and a cryostatic pressure assumption on the fourth boundary,
$$
\int_{\partial \Omega} \mathbf{T} \mathbf{n} \cdot \mathbf{v} \,d\Gamma = \int_{\partial \Omega} \frac{1}{2} \rho_i g H^2 \mathbf{n} \cdot \mathbf{v} \,d\Gamma,\tag{7}
$$
the weak form momentum equations read
$$
\int _{\Omega} \mathbf{T} : \nabla \mathbf{v}\,d\Omega + \int_{\Omega} \beta \mathbf{u} \cdot \mathbf{v} \,d\Omega = - \int_{\Omega}  \rho_i g H \nabla z_b \cdot \mathbf{v} \,d\Omega + \int_{\Omega} \frac{1}{2} \rho_i g H^2 \nabla \cdot \mathbf{v} \,d\Omega.\tag{8}
$$

## The thickness evolution equation

The geometry evolution is governed by the thickness evolution equation,
$$
\frac{\partial H}{\partial t} = - \nabla \cdot (H \mathbf{u}) + a_s - a_b,\tag{9}
$$

The semi-implicit Euler time discretization reads
$$
H_{k+1} + \Delta t \nabla \cdot (H_{k+1} \mathbf{u}_k) = H_k + \Delta t (a_s - a_b),\tag{10}
$$
where $H_k$ is the ice thickness at time step $k$, $H_{k+1}$ is the ice thickness at the next time step, $k+1$, $a_s$ is the surface accumulation rate and $a_b$ is the melt rate. Eq. (9) is used in this model to evolve the geometry of the ice.

The explicit Euler time discretization reads
$$
H_{k+1} = H_k - \Delta t \nabla \cdot (H_k \mathbf{u}_k) + \Delta t (a_s - a_b).\tag{11}
$$
This formulation is adapted to construct the numerical stabilization scheme.

## The Thickness Stabilization Scheme (TSS)

Multiplying Eq. (8) by $2 H$, an expression is found for the ice squared ice thickness at the next time step, $H_{k+1}^{2}$. Using an explicit Euler time discretization, this reads
$$
H^2_{k+1} = H^2_k - 2 \Delta t H_k \nabla \cdot (H_k \mathbf{u}) + 2 \Delta t (a_s - a_b).\tag{12}
$$

We use this expression as well as the explicit discretisation of the thickness evolution equation to mimick treating the right-hand side of Eq. (5) implictly. The TSS-stabilized SSA equations therefore read

$$
\int _{\Omega} \mathbf{T}_k : \nabla \mathbf{v}\,d\Omega + \int_{\Omega} \beta \tilde{\mathbf{u}}_{k+1} \cdot \mathbf{v} \,d\Omega - \int_{\Omega}  \rho_i g \theta \Delta t \nabla \cdot (H_k \tilde{\mathbf{u}}_{k+1}) \nabla z_b \cdot \mathbf{v} \,d\Omega \\ 
    + \int_{\Omega} \rho_i g \theta \Delta t H_k \nabla \cdot (H_k \tilde{\mathbf{u}}_{k+1}) \nabla \cdot \mathbf{v} \,d\Omega 
$$
$$
= - \int_{\Omega}  \rho_i g H_k \nabla z_b \cdot \mathbf{v} \,d\Omega \\
    - \int_{\Omega}  \rho_i g \theta \Delta t (a_s - a_b) \nabla z_b \cdot \mathbf{v} \,d\Omega + \int_{\Omega} \frac{1}{2} \rho_i g H^2_k \nabla \cdot \mathbf{v} \,d\Omega \\
    + \int_{\Omega} \rho_i g \theta \Delta t H_k (a_s - a_b) \nabla \cdot \mathbf{v} \,d\Omega.\tag{13}
$$
where the parameter $\theta \in [0,1]$ allows the user to perform a simulation with ($\theta = 1$) or without ($\theta = 0$) TSS. Here $\tilde{\mathbf{u}}_{k+1}$ is an approximation of the velocity at the next time step, $k+1$.


In [ ]:
from fenics import *
import matplotlib.pyplot as plt
import numpy as np
import time
import os
from constants import *
from SSA_functions import *
from FEM_setup import *


In [ ]:
start = time.time()

for i in range(num_TS):
    change=100
    tol=1e-5
    iter_sim=0
    maxiter=200

    while change>tol and iter_sim<maxiter:
        iter_sim=iter_sim+1
        a =   4*dot(mu*u1.dx(0), v1.dx(0))*dx \
                + 2*dot(mu*u2.dx(1), v1.dx(0))*dx \
                + dot(mu*u1.dx(1)+mu*u2.dx(0), v1.dx(1))*dx \
                + 4*dot(mu*u2.dx(1), v2.dx(1))*dx \
                + 2*dot(mu*u1.dx(0), v2.dx(1))*dx \
                + dot(mu*u1.dx(1)+mu*u2.dx(0), v2.dx(0))*dx \
                + beta2 * inner(uvect, vvect) * dx \
                + theta * rhoi * g * dt * thick * div(thick * uvect) * div(vvect) * dx \
                - theta * rhoi * g * dt * div(thick * uvect) * inner(grad(zb), vvect) * dx
            
        L = - rhoi * g * thick * inner(grad(zb), vvect) * dx \
                - theta * rhoi * g * dt * (a_s - a_b) * inner(grad(zb), vvect) * dx \
                + 0.5 * rhoi * g * thick * thick * div(vvect) * dx \
                + theta * rhoi * g * dt * thick * (a_s - a_b) * div(vvect) * dx

        # Compute solution
        uvecold=uvec.copy(deepcopy=True)
        (uxold,uyold)=split(uvecold)
        solve(a == L, uvec, [bc1,bc2,bc3,bc4])
        mu=viscosity(ux,uy,thick)
        change = norm(uvec.vector()-uvecold.vector())/norm(uvec.vector())
        print(change)
        
    print("Solving thickness evolution now...")
    
    vel = as_vector([ux, uy])
    vnorm = sqrt(dot(vel, vel) + DOLFIN_EPS)
    mu_art = 0.1 * h * vnorm
    
    F = (
        thick_new * phi * dx \
        - thick * phi * dx \
        + dt * dot((ux * thick_new).dx(0), phi) * dx \
        + dt * dot((uy * thick_new).dx(1), phi) * dx \
        + dt * dot(( -a_s + a_b ), phi) * dx \
        # Artifical viscosity
        + dt * mu_art * dot(grad(thick_new), grad(phi)) * dx
    )

    H = Function(V)
    solve(lhs(F) == rhs(F), H)
    thick = interpolate(H, V)
    
    # Minimum thickness constraint:
    thick.vector().set_local(np.maximum(thick.vector().get_local(), 10.0))
    thick.vector().apply("insert")
    print("Finished solving thickness evolution...")
    print("Year: ", (i+1)*dt)
end = time.time()


In [ ]:
print("Runtime: {:.4f} seconds".format(end - start))

In [ ]:
plot_field(thick, label=r'$H$')

In [ ]:
plot_field(zs, label=r'$z_s$')

In [ ]:
plot_field(zb, label=r'$z_b$')

In [ ]:
plot_field(ux, label=r'$u_x$')

In [ ]:
plot_field(uy, label=r'$u_y$')

In [ ]:
plot_field(beta2, label=r'$\beta$')

In [ ]:
filename = f"Results/u_dt_{dt}_theta_{theta}_T_{T}_res_{nx}.xdmf"

with XDMFFile(filename) as f:
    f.write_checkpoint(uvec, "velocity", 0.0, XDMFFile.Encoding.HDF5, append=False)

In [ ]:
filename = f"Results/thickness_dt_{dt}_theta_{theta}_T_{T}_res_{nx}.xdmf"

with XDMFFile(filename) as f:
    f.write_checkpoint(thick, "thickness", 0.0, XDMFFile.Encoding.HDF5, append=False)